# 🎬 4K Video Upscaler v2.0 (Real-ESRGAN)

Upscale your videos up to **4K / 8K** on free Google Colab or locally using [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN).

**What's New in v2.0:**
- ✅ **Audio preserved** — no more silent upscaled videos
- ✅ **Smart auto-tiling** — prevents OOM crashes
- ✅ **Batch processing** — upscale entire folders
- ✅ **Resume support** — skip already-processed files
- ✅ **BT.709 color space** — accurate colors on all players
- ✅ **Per-frame error recovery** — one bad frame won't kill the job
- ✅ **Live FPS + ETA** — know exactly how long it will take
- ✅ **Dry-run mode** — validate setup before committing GPU hours
- ✅ **Web-optimized output** — faststart for instant streaming

**Models:** `RealESRGAN_x4plus`, `RealESRGAN_x4plus_anime_6B`, `realesr-animevideov3`, `RealESRNet_x4plus`, `RealESRGAN_x2plus`, `realesr-general-x4v3`

**Resolutions:** FHD, 2K, 4K, 8K, or custom multipliers (2x, 3x, 4x, etc.)

**GitHub:** [pareshmishra23/4k-genration](https://github.com/pareshmishra23/4k-genration)

---
**⚠️ Colab Users:** Change Runtime to GPU — `Runtime` → `Change runtime type` → `T4 GPU`.


In [ ]:
#@title 1. Setup (~1-2 minutes) { display-mode: "form" }
import os, sys, subprocess, pathlib
import importlib.util
import torch

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Device: {DEVICE.upper()}")
if DEVICE == 'cuda':
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {vram_gb:.1f} GB")

# --- Clone / update repo ---
REPO_DIR = '4k-genration'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/pareshmishra23/4k-genration.git
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# --- Install dependencies ---
!pip install -q -r requirements.txt

# --- Clone & install Real-ESRGAN ---
if not os.path.exists('Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git
    %cd Real-ESRGAN
    !pip install -q .
    %cd ..
else:
    print("Real-ESRGAN already present.")

# --- Auto-patch basicsr for torchvision >= 0.17 ---
def patch_basicsr():
    try:
        spec = importlib.util.find_spec("basicsr")
        if spec is None:
            return
        import basicsr
        bp = pathlib.Path(basicsr.__file__).parent
        dp = bp / "data" / "degradations.py"
        if dp.exists():
            content = dp.read_text()
            old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
            new = "from torchvision.transforms.functional import rgb_to_grayscale"
            if old in content:
                dp.write_text(content.replace(old, new))
                print("[INFO] Patched basicsr for torchvision compatibility")
    except Exception as e:
        print(f"[WARN] Basicsr patch skipped: {e}")

patch_basicsr()

# --- Verify ---
if not os.path.exists('upscale_video.py'):
    raise FileNotFoundError("upscale_video.py not found! Setup failed.")

print("\n✅ Setup complete! Ready to upscale.")


# 2. Input Selection

Choose how to provide the video file.


In [ ]:
#@title 2. Choose Input Method { display-mode: "form" }
import sys, os
IN_COLAB = 'google.colab' in sys.modules  # Self-contained fallback

input_method = "Upload" #@param ["Upload", "Google Drive", "Local Path"]
video_path = ""

if input_method == "Google Drive" and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/gdrive/')
    video_path = "/content/gdrive/MyDrive/video.mp4" #@param {type:"string"}
    print("Drive mounted.")
elif input_method == "Upload" and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        video_path = os.path.abspath(fn)
        print(f'Uploaded: {fn} ({len(uploaded[fn])/1024/1024:.1f} MB)')
elif input_method == "Local Path":
    video_path = "/path/to/your/video.mp4" #@param {type:"string"}
else:
    if not IN_COLAB:
        print("Upload/Drive are Colab-only. Use 'Local Path'.")
        video_path = "/path/to/your/video.mp4"

if video_path and os.path.exists(video_path):
    import cv2
    cap = cv2.VideoCapture(video_path)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f"\n📹 Input: {os.path.basename(video_path)}")
    print(f"   Resolution: {w}x{h} | FPS: {fps:.2f} | Frames: {frames}")
else:
    print(f"⚠️ File not found: {video_path}")


# 3. Upscale Configuration

Configure target resolution, model, quality, and processing options.


In [ ]:
#@title 3. Configure & Run { display-mode: "form" }
#@markdown ---
output_directory = "./output" #@param {type:"string"}
resolution = "4k (3840 x 2160)" #@param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "8k (7680 x 4320)", "2 x original", "3 x original", "4 x original"]
model = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3", "RealESRNet_x4plus", "RealESRGAN_x2plus", "realesr-general-x4v3"]
#@markdown ---
#@markdown **Performance & Quality**
tile_size = 0 #@param {type:"integer"}
crf = 18 #@param {type:"slider", min:10, max:30, step:1}
preset = "slow" #@param ["ultrafast", "superfast", "veryfast", "faster", "fast", "medium", "slow", "slower", "veryslow"]
#@markdown ---
#@markdown **Options**
dry_run = False #@param {type:"boolean"}
keep_temp = False #@param {type:"boolean"}

if not video_path or not os.path.exists(video_path):
    print("❌ Error: No valid input video. Go back to Step 2.")
else:
    cmd = [
        "python", "upscale_video.py",
        "--input", video_path,
        "--output", output_directory,
        "--resolution", resolution,
        "--model", model,
        "--tile", str(tile_size),
        "--crf", str(crf),
        "--preset", preset,
    ]
    if dry_run:
        cmd.append("--dry-run")
        print("🔍 DRY RUN — validating pipeline...\n")
    if keep_temp:
        cmd.append("--keep-temp")
    
    !{' '.join(f'"{c}"' if ' ' in c else c for c in cmd)}


# 4. Batch Processing (Optional)

Upscale **all videos** in a folder with `--resume` to skip already-done files.


In [ ]:
#@title 4. Batch Upscale Folder { display-mode: "form" }
#@markdown Provide a folder path containing multiple videos.
batch_input_folder = "./videos" #@param {type:"string"}
batch_output_folder = "./output" #@param {type:"string"}
batch_resolution = "4k (3840 x 2160)" #@param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)", "8k (7680 x 4320)", "2 x original", "3 x original", "4 x original"]
batch_model = "RealESRGAN_x4plus" #@param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3", "RealESRNet_x4plus", "RealESRGAN_x2plus", "realesr-general-x4v3"]
batch_crf = 18 #@param {type:"slider", min:10, max:30, step:1}
batch_preset = "slow" #@param ["ultrafast", "superfast", "veryfast", "faster", "fast", "medium", "slow", "slower", "veryslow"]

if os.path.isdir(batch_input_folder):
    cmd = [
        "python", "upscale_video.py",
        "--input", batch_input_folder,
        "--output", batch_output_folder,
        "--resolution", batch_resolution,
        "--model", batch_model,
        "--crf", str(batch_crf),
        "--preset", batch_preset,
        "--batch",
        "--resume",
    ]
    print(f"📁 Batch mode: processing all videos in {batch_input_folder}\n")
    !{' '.join(f'"{c}"' if ' ' in c else c for c in cmd)}
else:
    print(f"❌ Not a valid directory: {batch_input_folder}")


# 5. Download Results

Download your upscaled videos.


In [ ]:
#@title 5. Download Upscaled Video(s) { display-mode: "form" }
import sys, os, glob
IN_COLAB = 'google.colab' in sys.modules

import glob
from google.colab import files

search_dir = output_directory if 'output_directory' in globals() else "./output"
output_files = sorted(glob.glob(os.path.join(search_dir, "*_*upscaled*.mp4")), key=os.path.getctime, reverse=True)

if not output_files:
    print("No upscaled videos found. Check the output directory.")
else:
    print(f"Found {len(output_files)} upscaled video(s):\n")
    for i, f in enumerate(output_files[:5], 1):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  {i}. {os.path.basename(f)} ({size_mb:.1f} MB)")
    
    if IN_COLAB:
        download_choice = "Latest" #@param ["Latest", "All"]
        if download_choice == "Latest" and output_files:
            print(f"\n⬇️ Downloading latest: {os.path.basename(output_files[0])}...")
            files.download(output_files[0])
        elif download_choice == "All":
            for f in output_files:
                print(f"⬇️ Downloading: {os.path.basename(f)}...")
                files.download(f)
    else:
        print(f"\n💾 Files are saved at: {os.path.abspath(search_dir)}")


# 🛠️ Troubleshooting

| Issue | Solution |
|-------|----------|
| **Out of Memory (OOM)** | Lower `tile_size` manually (e.g., 128 or 256) or switch to CPU. |
| **Audio missing** | v2.0 preserves audio automatically. If still missing, check source has audio. |
| **Slow processing** | Use `preset=fast` or `preset=medium`. Lower CRF = slower but better quality. |
| **Colors look wrong** | v2.0 tags BT.709. If still off, try a different model. |
| **basicsr import error** | The setup cell auto-patches this. If it fails, restart runtime and re-run setup. |
| **File too big for Colab** | Use Google Drive input instead of upload. |

**Model Guide:**
- `RealESRGAN_x4plus` — Best for general photos/videos (default)
- `realesr-animevideov3` — Fastest for anime/cartoons
- `RealESRGAN_x4plus_anime_6B` — Higher quality anime
- `realesr-general-x4v3` — General content + noise reduction
- `RealESRGAN_x2plus` — Use when you only need 2× upscale
